# V12 Submission Temp: December Holdout

Self-contained submission notebook copied from the optimized submission version.
It trains on January through November and exports December recommendations without importing any local `.py` helpers.
Optuna stays at 1 trial with a wider search space in the notebook logic.

In [ ]:
import polars as pl
import numpy as np
import lightgbm as lgb
import optuna
import os
import gc
import re
import warnings
import pickle
from pathlib import Path
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize

# Enable global string cache to guarantee Categorical alignment across all dataframes
pl.enable_string_cache()
warnings.filterwarnings('ignore')

SEED = 42
USE_CF = True
TRAIN_SAMPLE_USERS = 60000
EXPORT_CHUNK_SIZE = 10000
CF_CHUNK_SIZE = 2000
T_PATH = '/kaggle/input/datasets/kinonquc/qkindataset2/transaction_full_2025.parquet'
I_PATH = '/kaggle/input/datasets/kinonquc/qkindataset2/items.parquet'

print("Loading Data & Mapping Strings to Integers...")

# 1. Create a global integer mapping for items to prevent massive String RAM usage
items_raw = pl.read_parquet(I_PATH)
item_mapping = items_raw.select('item_id').unique().with_row_index("item_int_id")

# Save global mapping dictionary to stream back original string IDs at the end
idx2item = dict(zip(item_mapping['item_int_id'], item_mapping['item_id']))
# Also save as a global pickle file in case of memory separation
with open("idx2item.pkl", "wb") as f_map:
    pickle.dump(idx2item, f_map)

# 2. Load items, join mapping, and drop the string ID
items_df = items_raw.join(item_mapping, on='item_id', how='left').drop('item_id').rename({'item_int_id': 'item_id'}).select([
    pl.col('item_id').cast(pl.Int32), # Lightweight integer!
    pl.col('category').cast(pl.Utf8),
    pl.col('category_l1').cast(pl.Utf8),
    pl.col('category_l2').cast(pl.Utf8),
    pl.col('category_l3').cast(pl.Utf8),
    pl.col('brand').cast(pl.Utf8),
    pl.col('size').cast(pl.Utf8)
])

# 3. Load transactions, join mapping, and drop the string ID
df_raw = pl.read_parquet(T_PATH).join(item_mapping, on='item_id', how='inner').drop('item_id').rename({'item_int_id': 'item_id'}).select([
    pl.col('customer_id').cast(pl.Int64), # Must keep as Int64 to avoid overflow
    pl.col('item_id').cast(pl.Int32),     # Lightweight integer!
    pl.col('quantity').cast(pl.Int32),
    pl.col('price').cast(pl.Float32),
    # Map location strings to physical Int16 immediately
    pl.col('location').cast(pl.Categorical).to_physical().cast(pl.Int16), 
    pl.col('updated_date').cast(pl.Datetime).alias('event_ts')
 ]).drop_nulls(subset=['item_id', 'customer_id']).with_columns([
    pl.col('event_ts').dt.month().alias('month').cast(pl.Int8),
    pl.col('event_ts').dt.weekday().alias('dow').cast(pl.Int8)
])

# Clean up temporary frames
del items_raw, item_mapping
gc.collect()

cat_cols = ['category', 'category_l1', 'category_l2', 'category_l3', 'brand']
for c in cat_cols:
    items_df = items_df.with_columns(pl.col(c).fill_null('Unknown'))
    top_vals = items_df[c].value_counts().sort('count', descending=True).head(254)[c].to_list()
    items_df = items_df.with_columns(
        pl.when(pl.col(c).is_in(top_vals)).then(pl.col(c)).otherwise(pl.lit('Other')).alias(c)
    )
    items_df = items_df.with_columns(pl.col(c).cast(pl.Categorical).to_physical().cast(pl.Int32).alias(f"{c}_id"))

def standardize_age(text):
    raw_text = str(text).strip()
    clean_text = raw_text.lower()
    if re.search(r'(\*|x\d|cm)', clean_text): return 0.5
    if re.search(r'\bb\d{2}\b', clean_text): return 18.0
    if 's17' in clean_text: return 1.0
    if '110' in clean_text: return 5.0
    if "không xác định" in clean_text or not clean_text: return -1.0

    diaper_map = {
        r'\bnb\b': 0.0, r'\bss\b': 0.0, r'\bsơ sinh\b': 0.0,
        r'\bs\b': 0.25, r'\bm\b': 0.6, r'\bl\b': 1.2,
        r'\bxl\b': 2.0, r'\bxxl\b': 3.5
    }
    for pattern, val in diaper_map.items():
        if re.search(pattern, clean_text): return val

    range_match = re.search(r'(\d+\.?\d*)\s*-\s*(\d+\.?\d*)', clean_text)
    if range_match:
        s, e = float(range_match.group(1)), float(range_match.group(2))
        avg = (s + e) / 2
        if any(x in clean_text for x in ['m', 'tháng']): return round(avg / 12, 3)
        return avg

    m_match = re.search(r'(\d+\.?\d*)\s*(m|tháng)', clean_text)
    if m_match: return round(float(m_match.group(1)) / 12, 3)
    y_match = re.search(r'(\d+\.?\d*)\s*(y|t|tuổi)', clean_text)
    if y_match: return float(y_match.group(1))

    pure_num = re.search(r'^(\d+)$', clean_text)
    if pure_num:
        val = float(pure_num.group(1))
        return round(val/12, 3) if val > 6 else val
    return -1.0

size_map = {row[0]: standardize_age(row[1]) for row in items_df.select(['item_id', 'size']).iter_rows()}
items_df = items_df.with_columns(pl.col('item_id').replace(size_map, default=-1.0).cast(pl.Float32).alias('item_age_proxy'))


In [ ]:
class V12Retriever:
    def __init__(self, history_df, items_df, enable_cf=True):
        self.history_df = history_df
        self.items_df = items_df
        self.max_ts = history_df['event_ts'].max()
        self.enable_cf = enable_cf
        
        # Source 1: Global/Local Hot
        self.global_top = history_df.filter(pl.col('event_ts') >= self.max_ts - pl.duration(days=14))\
            .group_by('item_id').len().sort('len', descending=True).head(150).select('item_id')
            
        self.local_heroes = history_df.filter(pl.col('event_ts') >= self.max_ts - pl.duration(days=60))\
            .group_by(['location', 'item_id']).len()\
            .sort(['location', 'len'], descending=[False, True])\
            .group_by('location').head(80)
            
        # Source 2: Replenishment Cycle (Mathematically proven fast formula)
        self.replenish = history_df.group_by(['customer_id', 'item_id']).agg([
            pl.col('event_ts').count().alias('buy_count'),
            pl.col('event_ts').min().alias('first_buy'),
            pl.col('event_ts').max().alias('last_buy')
        ]).filter(pl.col('buy_count') > 1)\
          .with_columns(((pl.col('last_buy') - pl.col('first_buy')).dt.total_days() / (pl.col('buy_count') - 1)).alias('avg_gap'))
        
        # Source 3: CF (SVD + I2I)
        self._build_cf()
        
    def _build_cf(self):
        hist = self.history_df.filter(pl.col('event_ts') >= self.max_ts - pl.duration(days=180))
        u_map = hist['customer_id'].unique()
        i_map = hist['item_id'].unique()
        
        u_df = pl.DataFrame({
            'customer_id': u_map,
            'u_idx': np.arange(len(u_map), dtype=np.int64)
        })
        i_df = pl.DataFrame({
            'item_id': i_map,
            'i_idx': np.arange(len(i_map), dtype=np.int32)
        })
        
        hist_indexed = hist.join(u_df, on='customer_id', how='inner').join(i_df, on='item_id', how='inner')
        
        rows = hist_indexed['u_idx'].to_numpy()
        cols = hist_indexed['i_idx'].to_numpy()
        data = np.ones(len(rows))
        
        self.mtx = csr_matrix((data, (rows, cols)), shape=(len(u_map), len(i_map)))
        
        self.u2idx = dict(zip(u_df['customer_id'], u_df['u_idx']))
        self.i2idx = dict(zip(i_df['item_id'], i_df['i_idx']))
        self.idx2i = i_map.to_list()
        
        n_comp = min(100, len(i_map) - 1)
        self.svd = TruncatedSVD(n_components=n_comp, random_state=SEED)
        self.u_emb = self.svd.fit_transform(self.mtx)
        self.i_emb = self.svd.components_.T
        
        # Build I2I Similarity Matrix
        norm_m = normalize(self.mtx, norm='l2', axis=0)
        self.i2i_sim = (norm_m.T.dot(norm_m)).astype(np.float32)
        self.i2i_sim.setdiag(0)

    def get_candidates(self, target_users):
        cands = {}
        
        # History & Replenishment
        hist_s = self.history_df.filter(pl.col('customer_id').is_in(target_users))
        cands['hist'] = hist_s.select(['customer_id', 'item_id']).unique()
        
        due = self.replenish.filter(pl.col('customer_id').is_in(target_users))\
            .with_columns((self.max_ts - pl.col('last_buy')).dt.total_days().alias('days_since'))\
            .filter(pl.col('days_since') >= pl.col('avg_gap') * 0.8)\
            .select(['customer_id', 'item_id'])
        cands['repl'] = due
        
        # Popularity
        cands['global'] = pl.DataFrame({'customer_id': target_users}).join(self.global_top.with_columns(pl.lit(1).alias('_k')), how='cross').drop('_k')
        
        user_loc = hist_s.group_by('customer_id').agg(pl.col('location').mode().first().alias('location'))
        cands['local'] = user_loc.join(self.local_heroes, on='location').select(['customer_id', 'item_id']).unique()
        
        # CF Chunks (Vectorized SVD & I2I)
        u_idx = [self.u2idx[u] for u in target_users if u in self.u2idx]
        t_u = [u for u in target_users if u in self.u2idx]
        i_arr = np.array(self.idx2i)
        if u_idx:
            chunk = 4000
            c_svd, c_i2i = [], []
            for i in range(0, len(u_idx), chunk):
                idx_chunk = u_idx[i:i+chunk]
                u_b = np.array(t_u[i:i+chunk])
                # CF (SVD)
                scores_svd = self.u_emb[idx_chunk] @ self.i_emb.T
                t60 = np.argsort(-scores_svd, axis=1)[:, :60]
                c_svd.append(pl.DataFrame({
                    'customer_id': pl.Series(np.repeat(u_b, 60), dtype=pl.Int64),
                    'item_id': i_arr[t60.flatten()]
                }))
                # CF (I2I)
                scores_i2i = self.mtx[idx_chunk].dot(self.i2i_sim).toarray()
                t80 = np.argsort(-scores_i2i, axis=1)[:, :80]
                mask = np.take_along_axis(scores_i2i, t80, axis=1) > 0
                c_i2i.append(pl.DataFrame({
                    'customer_id': pl.Series(np.repeat(u_b, 80)[mask.flatten()], dtype=pl.Int64),
                    'item_id': i_arr[t80.flatten()][mask.flatten()]
                }))
            cands['svd'] = pl.concat(c_svd).unique() if c_svd else pl.DataFrame(schema={'customer_id': pl.Int64, 'item_id': pl.Utf8})
            cands['i2i'] = pl.concat(c_i2i).unique() if c_i2i else pl.DataFrame(schema={'customer_id': pl.Int64, 'item_id': pl.Utf8})
            
        # Association & Category Top
        u_cat_top = self.history_df.filter(pl.col('customer_id').is_in(target_users))\
            .join(self.items_df.select(['item_id', 'category_l1']), on='item_id')\
            .group_by(['customer_id', 'category_l1']).len().sort('len', descending=True).group_by('customer_id').head(1)
        
        cat_global_top = self.history_df.filter(pl.col('event_ts') >= self.max_ts - pl.duration(days=30))\
            .join(self.items_df.select(['item_id', 'category_l1']), on='item_id')\
            .group_by(['category_l1', 'item_id']).len().sort('len', descending=True).group_by('category_l1').head(10)
            
        cands['cat_top'] = u_cat_top.join(cat_global_top, on='category_l1').select(['customer_id', 'item_id'])

        all_c = pl.concat([df for df in cands.values() if df is not None and df.height > 0]).unique()
        return all_c


In [ ]:
def create_dataset_v12(history_df, truth_df, items_df, sample_users=None, n_negatives=150, target_users=None, retriever=None):
    if target_users is not None:
        valid_u = list(target_users)
    elif sample_users:
        valid_u = history_df['customer_id'].unique().shuffle(seed=SEED).head(sample_users).to_list()
    else:
        valid_u = history_df['customer_id'].unique().to_list()
    
    if retriever is None:
        retriever = V12Retriever(history_df, items_df)
    ds = retriever.get_candidates(valid_u)
    
    if truth_df is not None:
        truth = truth_df.filter(pl.col('customer_id').is_in(valid_u)).select(['customer_id', 'item_id']).unique()
        ds = ds.join(truth.with_columns(pl.lit(1).cast(pl.Int8).alias('target')), on=['customer_id', 'item_id'], how='left').fill_null(0)
        missed = truth.join(ds, on=['customer_id', 'item_id'], how='anti').with_columns(pl.lit(1).cast(pl.Int8).alias('target'))
        ds = pl.concat([ds, missed]).unique(subset=['customer_id', 'item_id'])
        if n_negatives:
            pos = ds.filter(pl.col('target') == 1)
            neg = ds.filter(pl.col('target') == 0).sample(fraction=1.0, shuffle=True, seed=SEED).group_by('customer_id').head(n_negatives)
            ds = pl.concat([pos, neg])
        ds = ds.sort(['customer_id', 'target'], descending=[False, True])
    else:
        ds = ds.sort('customer_id')
    
    # --- FEATURES: THE CANNONS (STRICTLY DATA-DRIVEN) ---
    max_ts = history_df['event_ts'].max()
    
    # 1. User Profile Features
    # Brand commitment (Idea 2): calculate brand loyalty/HHI per customer
    u_brand_counts = history_df.join(items_df.select(['item_id', 'brand']), on='item_id')\
        .group_by(['customer_id', 'brand']).len().rename({'len': 'brand_count'})
    u_brand_hhi = u_brand_counts.with_columns(
        (pl.col('brand_count') / pl.col('brand_count').sum().over('customer_id')).alias('brand_share')
    ).with_columns(
        (pl.col('brand_share') * pl.col('brand_share')).alias('brand_share_sq')
    ).group_by('customer_id').agg(pl.col('brand_share_sq').sum().alias('u_brand_hhi'))

    # Category Affinity HHI Specialization (Idea 42)
    u_cat_counts = history_df.join(items_df.select(['item_id', 'category_l1']), on='item_id')\
        .group_by(['customer_id', 'category_l1']).len().rename({'len': 'cat_count'})
    u_cat_hhi = u_cat_counts.with_columns(
        (pl.col('cat_count') / pl.col('cat_count').sum().over('customer_id')).alias('cat_share')
    ).with_columns(
        (pl.col('cat_share') * pl.col('cat_share')).alias('cat_share_sq')
    ).group_by('customer_id').agg(pl.col('cat_share_sq').sum().alias('u_cat_hhi'))

    # User size age-proxy preference profiling (Idea 22, 34, 45)
    global_avg_age = items_df.filter(pl.col('item_age_proxy') >= 0)['item_age_proxy'].mean()
    if global_avg_age is None:
        global_avg_age = 1.0
    u_avg_age = history_df.join(items_df.select(['item_id', 'item_age_proxy']), on='item_id')\
        .filter(pl.col('item_age_proxy') >= 0)\
        .group_by('customer_id').agg(pl.col('item_age_proxy').mean().alias('u_avg_age_proxy'))

    u_prof = history_df.group_by('customer_id').agg([
        pl.col('item_id').n_unique().alias('u_unique_items'),
        pl.col('quantity').sum().alias('u_total_qty'),
        pl.col('price').mean().alias('u_avg_price'),
        pl.col('price').std().alias('u_price_std'),
        (max_ts - pl.col('event_ts').min()).dt.total_days().alias('u_tenure_days'),
        (pl.col('item_id').n_unique() / pl.col('quantity').sum().clip(1)).alias('u_exploration_ratio')
    ]).join(u_brand_hhi, on='customer_id', how='left')\
      .join(u_cat_hhi, on='customer_id', how='left')\
      .join(u_avg_age, on='customer_id', how='left')\
      .with_columns(pl.col('u_avg_age_proxy').fill_null(global_avg_age))
    
    # 2. Item Profile Features
    # Item repeat propensity (Idea 17, 44)
    i_repeats = history_df.group_by(['item_id', 'customer_id']).len().filter(pl.col('len') > 1)\
        .group_by('item_id').len().rename({'len': 'repeat_buyers'})
    
    i_prof = history_df.group_by('item_id').agg([
        pl.col('customer_id').n_unique().alias('i_unique_users'),
        pl.col('quantity').sum().alias('i_total_qty'),
        pl.col('location').n_unique().alias('i_hubs_count'),
        pl.col('price').median().alias('i_ref_price')
    ]).join(i_repeats, on='item_id', how='left')\
      .with_columns((pl.col('repeat_buyers').fill_null(0) / pl.col('i_unique_users')).alias('i_repeat_rate'))\
      .drop('repeat_buyers')
    
    # 3. User-Item Features
    ui_hist = history_df.filter(pl.col('customer_id').is_in(valid_u)).group_by(['customer_id', 'item_id']).agg([
        pl.col('quantity').sum().alias('ui_total_qty'),
        (max_ts - pl.col('event_ts').max()).dt.total_days().alias('ui_recency_days')
    ])
    
    # Preferred category (Idea 42) & Preferred brand (Idea 2, 43)
    u_pref_cat = history_df.filter(pl.col('customer_id').is_in(valid_u))\
        .join(items_df.select(['item_id', 'category_l1']), on='item_id')\
        .group_by(['customer_id', 'category_l1']).len().sort('len', descending=True)\
        .group_by('customer_id').head(1).select(['customer_id', 'category_l1']).rename({'category_l1': 'pref_cat_l1'})
        
    u_pref_brand = history_df.filter(pl.col('customer_id').is_in(valid_u))\
        .join(items_df.select(['item_id', 'category_l1', 'brand']), on='item_id')\
        .group_by(['customer_id', 'category_l1', 'brand']).len().sort('len', descending=True)\
        .group_by(['customer_id', 'category_l1']).head(1).select(['customer_id', 'category_l1', 'brand']).rename({'brand': 'pref_brand'})

    # Momentum (Idea 23)
    vol_7d = history_df.filter(pl.col('event_ts') >= max_ts - pl.duration(days=7)).group_by('item_id').len().rename({'len': 'v7'})
    vol_21d = history_df.filter(pl.col('event_ts') >= max_ts - pl.duration(days=21)).group_by('item_id').len().rename({'len': 'v21'})
    momentum = vol_7d.join(vol_21d, on='item_id', how='left').with_columns((pl.col('v7') / (pl.col('v21') / 3.0 + 1)).alias('item_momentum'))
    
    # Category Affinity
    u_cat = history_df.join(items_df.select(['item_id', 'category_l1']), on='item_id')\
        .group_by(['customer_id', 'category_l1']).len()\
        .with_columns((pl.col('len') / pl.col('len').sum().over('customer_id')).alias('u_cat_affinity'))
    
    ds = ds.join(u_prof, on='customer_id', how='left')
    ds = ds.join(i_prof, on='item_id', how='left')
    ds = ds.join(ui_hist, on=['customer_id', 'item_id'], how='left')
    ds = ds.join(items_df.select(['item_id', 'item_age_proxy', 'brand', 'category_l1'] + [f'{c}_id' for c in cat_cols]), on='item_id', how='left')
    ds = ds.join(momentum.select(['item_id', 'item_momentum']), on='item_id', how='left')
    ds = ds.join(u_cat.select(['customer_id', 'category_l1', 'u_cat_affinity']), on=['customer_id', 'category_l1'], how='left')
    
    # Joins and Calculations for Preferred Category and Brand
    ds = ds.join(u_pref_cat, on='customer_id', how='left')
    ds = ds.join(u_pref_brand, on=['customer_id', 'category_l1'], how='left')
    
    ds = ds.with_columns([
        pl.when(pl.col('category_l1') == pl.col('pref_cat_l1')).then(1).otherwise(0).alias('ui_is_primary_cat'),
        pl.when(pl.col('brand') == pl.col('pref_brand')).then(1).otherwise(0).alias('ui_is_preferred_brand')
    ]).drop(['pref_cat_l1', 'pref_brand', 'brand'])
    
    # 5. Price Sensitivity & Alignment Features (Idea 10, 35, 48)
    ds = ds.with_columns([
        (pl.col('i_ref_price') - pl.col('u_avg_price')).abs().alias('ui_price_diff'),
        (pl.col('i_ref_price') / (pl.col('u_avg_price') + 1e-5)).alias('ui_price_ratio')
    ])
    
    # 6. Location Availability & Assortment Gap Features (Idea 36, 38)
    u_loc = history_df.group_by('customer_id').agg(pl.col('location').mode().first().alias('location'))
    loc_item_pop = history_df.group_by(['location', 'item_id']).len().rename({'len': 'ui_loc_sales'})
    ds = ds.join(u_loc, on='customer_id', how='left')
    ds = ds.join(loc_item_pop, on=['location', 'item_id'], how='left').drop('location')
    
    # 7. NEW CHAMPIONSHIP UPGRADES (Idea 34, 36, 38, 42, 44, 45)
    ds = ds.with_columns([
        # Size Progression differences (Idea 22, 34, 45)
        (pl.col('item_age_proxy') - pl.col('u_avg_age_proxy')).alias('ui_size_age_diff'),
        (pl.col('item_age_proxy') / (pl.col('u_avg_age_proxy') + 1e-5)).alias('ui_size_age_ratio'),
        # Non-consumable discretionary repeat penalty flag (Idea 44)
        pl.when(pl.col('category_l1').is_in(['Thời trang', 'Đồ chơi & Sách', 'Phụ kiện']) & pl.col('ui_total_qty').is_not_null())\
          .then(1).otherwise(0).alias('ui_already_bought_discretionary'),
        # Assortment Ghost SKU penalty flag (Idea 36, 38)
        pl.when(pl.col('category_l1').is_in(['Thời trang', 'Đồ chơi & Sách', 'Phụ kiện']) & (pl.col('ui_loc_sales') == 0))\
          .then(1).otherwise(0).alias('ui_loc_sparsity_penalty')
    ])

    # Safe numerical-only fill_null to prevent Categorical column crash
    num_cols = [c for c in ds.columns if c not in ['customer_id', 'item_id', 'category_l1', 'target']]
    ds = ds.with_columns([
        pl.col(num_cols).fill_null(0)
    ]).drop('category_l1')
    
    return ds.sort(['customer_id', 'target'], descending=[False, True]) if truth_df is not None else ds.sort('customer_id')


In [ ]:
def create_dataset_v12(history_df, truth_df, items_df, sample_users=None, n_negatives=150, target_users=None, retriever=None):
    if target_users is not None:
        valid_u = list(target_users)
    elif sample_users:
        valid_u = history_df['customer_id'].unique().shuffle(seed=SEED).head(sample_users).to_list()
    else:
        valid_u = history_df['customer_id'].unique().to_list()
    
    if retriever is None:
        retriever = V12Retriever(history_df, items_df)
    ds = retriever.get_candidates(valid_u)
    
    has_target = truth_df is not None
    if has_target:
        truth = truth_df.filter(pl.col('customer_id').is_in(valid_u)).select(['customer_id', 'item_id']).unique()
        ds = ds.join(truth.with_columns(pl.lit(1).cast(pl.Int8).alias('target')), on=['customer_id', 'item_id'], how='left').fill_null(0)
        missed = truth.join(ds, on=['customer_id', 'item_id'], how='anti').with_columns(pl.lit(1).cast(pl.Int8).alias('target'))
        ds = pl.concat([ds, missed]).unique(subset=['customer_id', 'item_id'])
        if n_negatives:
            pos = ds.filter(pl.col('target') == 1)
            neg = ds.filter(pl.col('target') == 0).sample(fraction=1.0, shuffle=True, seed=SEED).group_by('customer_id').head(n_negatives)
            ds = pl.concat([pos, neg])
        ds = ds.sort(['customer_id', 'target'], descending=[False, True])
    else:
        ds = ds.sort('customer_id')
    
    # --- FEATURES: THE CANNONS (STRICTLY DATA-DRIVEN) ---
    max_ts = history_df['event_ts'].max()
    
    # 1. User Profile Features
    # Brand commitment (Idea 2): calculate brand loyalty/HHI per customer
    u_brand_counts = history_df.join(items_df.select(['item_id', 'brand']), on='item_id')\
        .group_by(['customer_id', 'brand']).len().rename({'len': 'brand_count'})
    u_brand_hhi = u_brand_counts.with_columns(
        (pl.col('brand_count') / pl.col('brand_count').sum().over('customer_id')).alias('brand_share')
    ).with_columns(
        (pl.col('brand_share') * pl.col('brand_share')).alias('brand_share_sq')
    ).group_by('customer_id').agg(pl.col('brand_share_sq').sum().alias('u_brand_hhi'))

    # Category Affinity HHI Specialization (Idea 42)
    u_cat_counts = history_df.join(items_df.select(['item_id', 'category_l1']), on='item_id')\
        .group_by(['customer_id', 'category_l1']).len().rename({'len': 'cat_count'})
    u_cat_hhi = u_cat_counts.with_columns(
        (pl.col('cat_count') / pl.col('cat_count').sum().over('customer_id')).alias('cat_share')
    ).with_columns(
        (pl.col('cat_share') * pl.col('cat_share')).alias('cat_share_sq')
    ).group_by('customer_id').agg(pl.col('cat_share_sq').sum().alias('u_cat_hhi'))

    # User size age-proxy preference profiling (Idea 22, 34, 45)
    global_avg_age = items_df.filter(pl.col('item_age_proxy') >= 0)['item_age_proxy'].mean()
    if global_avg_age is None:
        global_avg_age = 1.0
    u_avg_age = history_df.join(items_df.select(['item_id', 'item_age_proxy']), on='item_id')\
        .filter(pl.col('item_age_proxy') >= 0)\
        .group_by('customer_id').agg(pl.col('item_age_proxy').mean().alias('u_avg_age_proxy'))

    u_prof = history_df.group_by('customer_id').agg([
        pl.col('item_id').n_unique().alias('u_unique_items'),
        pl.col('quantity').sum().alias('u_total_qty'),
        pl.col('price').mean().alias('u_avg_price'),
        pl.col('price').std().alias('u_price_std'),
        (max_ts - pl.col('event_ts').min()).dt.total_days().alias('u_tenure_days'),
        (pl.col('item_id').n_unique() / pl.col('quantity').sum().clip(1)).alias('u_exploration_ratio')
    ]).join(u_brand_hhi, on='customer_id', how='left')\
      .join(u_cat_hhi, on='customer_id', how='left')\
      .join(u_avg_age, on='customer_id', how='left')\
      .with_columns(pl.col('u_avg_age_proxy').fill_null(global_avg_age))
    
    # 2. Item Profile Features
    # Item repeat propensity (Idea 17, 44)
    i_repeats = history_df.group_by(['item_id', 'customer_id']).len().filter(pl.col('len') > 1)\
        .group_by('item_id').len().rename({'len': 'repeat_buyers'})
    
    i_prof = history_df.group_by('item_id').agg([
        pl.col('customer_id').n_unique().alias('i_unique_users'),
        pl.col('quantity').sum().alias('i_total_qty'),
        pl.col('location').n_unique().alias('i_hubs_count'),
        pl.col('price').median().alias('i_ref_price')
    ]).join(i_repeats, on='item_id', how='left')\
      .with_columns((pl.col('repeat_buyers').fill_null(0) / pl.col('i_unique_users')).alias('i_repeat_rate'))\
      .drop('repeat_buyers')
    
    # 3. User-Item Features
    ui_hist = history_df.filter(pl.col('customer_id').is_in(valid_u)).group_by(['customer_id', 'item_id']).agg([
        pl.col('quantity').sum().alias('ui_total_qty'),
        (max_ts - pl.col('event_ts').max()).dt.total_days().alias('ui_recency_days')
    ])
    
    # Preferred category (Idea 42) & Preferred brand
    u_pref_cat = history_df.filter(pl.col('customer_id').is_in(valid_u))\
        .join(items_df.select(['item_id', 'category_l1']), on='item_id')\
        .group_by(['customer_id', 'category_l1']).len().sort('len', descending=True)\
        .group_by('customer_id').head(1).select(['customer_id', 'category_l1']).rename({'category_l1': 'pref_cat_l1'})
        
    u_pref_brand = history_df.filter(pl.col('customer_id').is_in(valid_u))\
        .join(items_df.select(['item_id', 'category_l1', 'brand']), on='item_id')\
        .group_by(['customer_id', 'category_l1', 'brand']).len().sort('len', descending=True)\
        .group_by(['customer_id', 'category_l1']).head(1).select(['customer_id', 'category_l1', 'brand']).rename({'brand': 'pref_brand'})

    # Momentum (Idea 23)
    vol_7d = history_df.filter(pl.col('event_ts') >= max_ts - pl.duration(days=7)).group_by('item_id').len().rename({'len': 'v7'})
    vol_21d = history_df.filter(pl.col('event_ts') >= max_ts - pl.duration(days=21)).group_by('item_id').len().rename({'len': 'v21'})
    momentum = vol_7d.join(vol_21d, on='item_id', how='left').with_columns((pl.col('v7') / (pl.col('v21') / 3.0 + 1)).alias('item_momentum'))
    
    # Category Affinity
    u_cat = history_df.join(items_df.select(['item_id', 'category_l1']), on='item_id')\
        .group_by(['customer_id', 'category_l1']).len()\
        .with_columns((pl.col('len') / pl.col('len').sum().over('customer_id')).alias('u_cat_affinity'))
    
    ds = ds.join(u_prof, on='customer_id', how='left')
    ds = ds.join(i_prof, on='item_id', how='left')
    ds = ds.join(ui_hist, on=['customer_id', 'item_id'], how='left')
    ds = ds.join(items_df.select(['item_id', 'item_age_proxy', 'brand', 'category_l1'] + [f'{c}_id' for c in cat_cols]), on='item_id', how='left')
    ds = ds.join(momentum.select(['item_id', 'item_momentum']), on='item_id', how='left')
    ds = ds.join(u_cat.select(['customer_id', 'category_l1', 'u_cat_affinity']), on=['customer_id', 'category_l1'], how='left')
    
    ds = ds.join(u_pref_cat, on='customer_id', how='left')
    ds = ds.join(u_pref_brand, on=['customer_id', 'category_l1'], how='left')
    
    ds = ds.with_columns([
        pl.when(pl.col('category_l1') == pl.col('pref_cat_l1')).then(1).otherwise(0).alias('ui_is_primary_cat'),
        pl.when(pl.col('brand') == pl.col('pref_brand')).then(1).otherwise(0).alias('ui_is_preferred_brand')
    ]).drop(['pref_cat_l1', 'pref_brand', 'brand'])
    
    ds = ds.with_columns([
        (pl.col('i_ref_price') - pl.col('u_avg_price')).abs().alias('ui_price_diff'),
        (pl.col('i_ref_price') / (pl.col('u_avg_price') + 1e-5)).alias('ui_price_ratio')
    ])
    
    u_loc = history_df.group_by('customer_id').agg(pl.col('location').mode().first().alias('location'))
    loc_item_pop = history_df.group_by(['location', 'item_id']).len().rename({'len': 'ui_loc_sales'})
    ds = ds.join(u_loc, on='customer_id', how='left')
    ds = ds.join(loc_item_pop, on=['location', 'item_id'], how='left').drop('location')
    ds = ds.with_columns(pl.col('ui_loc_sales').fill_null(0))
    
    ds = ds.with_columns([
        (pl.col('item_age_proxy') - pl.col('u_avg_age_proxy')).abs().alias('ui_size_age_diff'),
        (pl.col('item_age_proxy') / (pl.col('u_avg_age_proxy') + 1e-5)).alias('ui_size_age_ratio'),
        pl.when(pl.col('category_l1').is_in(['Thời trang', 'Đồ chơi & Sách', 'Phụ kiện']) & (pl.col('ui_total_qty').fill_null(0) > 0))\
          .then(1).otherwise(0).alias('ui_already_bought_discretionary'),
        pl.when(pl.col('category_l1').is_in(['Thời trang', 'Đồ chơi & Sách', 'Phụ kiện']) & (pl.col('ui_loc_sales') == 0))\
          .then(1).otherwise(0).alias('ui_loc_sparsity_penalty')
    ])

    num_cols = [c for c in ds.columns if c not in ['customer_id', 'item_id', 'category_l1', 'target']]
    ds = ds.with_columns([
        pl.col(num_cols).fill_null(0)
    ]).drop('category_l1')
    
    if has_target:
        return ds.sort(['customer_id', 'target'], descending=[False, True])
    return ds.sort('customer_id')

In [ ]:
print("Preparing Folds...")

PINNED_LGB_PARAMS = {
    'objective': 'lambdarank',
    'metric': 'ndcg',
    'ndcg_eval_at': [10],
    'verbosity': -1,
    'learning_rate': 0.012877754210689124,
    'num_leaves': 494,
    'max_depth': 13,
    'min_data_in_leaf': 326,
    'lambda_l1': 0.0005425710423504571,
    'lambda_l2': 9.669321423455777e-07,
    'max_bin': 255,
    'device': 'gpu',
    'random_state': SEED,
}

FIXED_NUM_BOOST_ROUND = 800
FIXED_EARLY_STOPPING_ROUNDS = 50
TRAIN_FINAL_NUM_BOOST_ROUND = 1200

def get_fold(train_end, val_m):
    h = df_raw.filter(pl.col('month') <= train_end)
    t = df_raw.filter(pl.col('month') == val_m)
    return create_dataset_v12(h, t, items_df, sample_users=TRAIN_SAMPLE_USERS, n_negatives=150)

f1 = get_fold(8, 9)
f2 = get_fold(9, 10)
f3 = get_fold(10, 11)

cat_feat_names = [f'{c}_id' for c in cat_cols]
all_feats = [
    'u_unique_items', 'u_total_qty', 'u_avg_price', 'u_price_std', 'u_tenure_days', 'u_exploration_ratio', 'u_brand_hhi',
    'i_unique_users', 'i_total_qty', 'i_hubs_count', 'i_ref_price', 'i_repeat_rate',
    'ui_total_qty', 'ui_recency_days', 'ui_is_primary_cat', 'ui_is_preferred_brand',
    'ui_price_diff', 'ui_price_ratio', 'ui_loc_sales', 'item_momentum', 'item_age_proxy', 'u_cat_affinity',
    'u_cat_hhi', 'u_avg_age_proxy', 'ui_size_age_diff', 'ui_size_age_ratio', 'ui_already_bought_discretionary', 'ui_loc_sparsity_penalty'
] + cat_feat_names
cat_feat_ids = [all_feats.index(c) for c in cat_feat_names]

def prep_lgb(df):
    feat_frame = df.select(all_feats)
    x = feat_frame.to_numpy().astype(np.float32, copy=False)
    y = df.get_column('target').to_numpy().astype(np.int8, copy=False)
    g = df.group_by('customer_id', maintain_order=True).len().get_column('len').to_numpy().astype(np.int32, copy=False)
    return x, y, g

X1, y1, g1 = prep_lgb(f1)
X2, y2, g2 = prep_lgb(f2)
X3, y3, g3 = prep_lgb(f3)

def train_fixed_model(X_train, y_train, g_train, X_val, y_val, g_val):
    dtrain = lgb.Dataset(X_train, y_train, group=g_train, categorical_feature=cat_feat_ids)
    dval = lgb.Dataset(X_val, y_val, group=g_val, reference=dtrain, categorical_feature=cat_feat_ids)
    booster = lgb.train(
        PINNED_LGB_PARAMS,
        dtrain,
        valid_sets=[dval],
        num_boost_round=FIXED_NUM_BOOST_ROUND,
        callbacks=[lgb.early_stopping(FIXED_EARLY_STOPPING_ROUNDS), lgb.log_evaluation(25)],
    )
    return booster

X_train = np.vstack([X1, X2])
y_train = np.concatenate([y1, y2])
g_train = np.concatenate([g1, g2])

lgb_m = train_fixed_model(X_train, y_train, g_train, X3, y3, g3)
print('Pinned params:', PINNED_LGB_PARAMS)
print('best_iteration:', lgb_m.best_iteration)

X_final = np.vstack([X1, X2, X3])
y_final = np.concatenate([y1, y2, y3])
g_final = np.concatenate([g1, g2, g3])
d_final = lgb.Dataset(X_final, y_final, group=g_final, categorical_feature=cat_feat_ids)
lgb_m = lgb.train(PINNED_LGB_PARAMS, d_final, num_boost_round=lgb_m.best_iteration or TRAIN_FINAL_NUM_BOOST_ROUND)

del X1, y1, g1, X2, y2, g2, X3, y3, g3, X_train, y_train, g_train, X_final, y_final, g_final, d_final, f1, f2, f3
gc.collect()


In [ ]:
from pathlib import Path
import json
import pickle


def make_chunked_submission_pkl(history_df, items_df, target_users, model, all_feats, output_path, chunk_size=EXPORT_CHUNK_SIZE, write_chunk_files=False):
    """Build the final submission as a customer_id -> [item_id, ...] mapping.

    The pipeline still processes users in chunks for safety, but the final artifact
    written to `output_path` is streamed directly to pickle so we do not keep the full
    submission dict in memory. Each customer gets exactly 10 items. If fewer than 10
    candidates survive, the list is cycled or padded from a global fallback list.
    Intermediate chunk pickle writes are disabled by default because they dominate
    wall-clock time and are unnecessary for the Kaggle submission artifact.
    """
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    chunk_dir = output_path.parent / f"{output_path.stem}_chunks"
    if write_chunk_files:
        chunk_dir.mkdir(parents=True, exist_ok=True)

    user_list = list(target_users)
    total_users = len(user_list)
    chunk_files = []
    prediction_cols = None
    written_users = 0
    output_handle = None

    retriever = V12Retriever(history_df, items_df, enable_cf=USE_CF)

    def to_original_item_id(item_id):
        item_int = int(item_id)
        return str(idx2item.get(item_int, item_id)).zfill(13)

    def to_original_item_ids(items):
        return [to_original_item_id(item) for item in items]

    def to_customer_key(customer_id):
        return int(customer_id)

    global_fallback = (
        history_df.group_by('item_id').len()
        .sort('len', descending=True)
        .select('item_id')
        .to_series()
        .to_list()
    )
    if not global_fallback:
        global_fallback = items_df.get_column('item_id').head(10).to_list()
    global_fallback = to_original_item_ids(global_fallback)

    def ensure_exact_10(items):
        values = list(items)
        if not values:
            values = list(global_fallback[:10])
        if not values:
            return []
        if len(values) >= 10:
            return values[:10]
        needed = 10 - len(values)
        repeated = (values * ((needed // len(values)) + 2))[:needed]
        return values + repeated

    def write_pickle_record(handle, key, value):
        handle.write(pickle.dumps(key, protocol=pickle.HIGHEST_PROTOCOL)[2:-1])
        handle.write(pickle.dumps(value, protocol=pickle.HIGHEST_PROTOCOL)[2:-1])
        handle.write(b's')

    with open(output_path, 'wb') as output_handle:
        output_handle.write(b'\x80\x04}')
        for chunk_num, start in enumerate(range(0, total_users, chunk_size), 1):
            end = min(start + chunk_size, total_users)
            chunk_users = user_list[start:end]
            print(f"[chunk {chunk_num}] users {start:,}..{end - 1:,} ({len(chunk_users):,})")

            ds = create_dataset_v12(
                history_df,
                None,
                items_df,
                target_users=chunk_users,
                n_negatives=150,
                retriever=retriever,
            )
            if ds.is_empty():
                print("  -> skipped: no candidates")
                chunk_recs = {to_customer_key(uid): ensure_exact_10([]) for uid in chunk_users}
            else:
                if prediction_cols is None:
                    missing = [c for c in all_feats if c not in ds.columns]
                    if missing:
                        raise ValueError(f'Missing prediction features in submission chunk: {missing}')
                    prediction_cols = all_feats
                X_sub = ds.select(prediction_cols).to_numpy().astype(np.float32, copy=False)
                preds = model.predict(X_sub, num_iteration=model.best_iteration)
                ds = ds.with_columns(pl.Series('pred', preds))

                top10 = (
                    ds
                    .group_by('customer_id', maintain_order=True)
                    .agg(pl.col('item_id').sort_by('pred', descending=True).alias('ranked_items'))
                )

                chunk_recs = {to_customer_key(customer_id): ensure_exact_10(to_original_item_ids(ranked_items)) for customer_id, ranked_items in top10.iter_rows()}
                missing_users = {to_customer_key(uid) for uid in chunk_users} - set(chunk_recs.keys())
                for customer_id in missing_users:
                    chunk_recs[customer_id] = ensure_exact_10([])

                del ds, X_sub, preds, top10

            if write_chunk_files:
                chunk_file = chunk_dir / f"submission_chunk_{chunk_num:04d}.pkl"
                with open(chunk_file, 'wb') as f:
                    pickle.dump(chunk_recs, f, protocol=pickle.HIGHEST_PROTOCOL)
                chunk_files.append(str(chunk_file))
                print(f"  -> wrote {len(chunk_recs):,} users to {chunk_file.name}")
            else:
                print(f"  -> processed {len(chunk_recs):,} users")

            for customer_id, recs in chunk_recs.items():
                write_pickle_record(output_handle, customer_id, recs)
            written_users += len(chunk_recs)
            del chunk_recs
            gc.collect()

        output_handle.write(b'.')

    manifest = output_path.with_name(f"{output_path.stem}_manifest.json")
    manifest.write_text(
        json.dumps(
            {
                'output_path': str(output_path),
                'chunks': chunk_files,
                'total_users': total_users,
                'written_users': written_users,
                'format': 'pickle-dict-customer_id-to-item_list',
                'top_k': 10,
                'duplicates_preserved': True,
                'exact_length_per_customer': 10,
                'chunk_file_writes': write_chunk_files,
                'streamed_pickle': True,
                'chunk_size': chunk_size,
            },
            indent=2,
        ),
        encoding='utf-8',
    )

    print(f"Final submission: {output_path}")
    print(f"Users written: {written_users:,} / {total_users:,}")
    print(f"Manifest: {manifest}")
    return output_path


output_file = Path('submission.pkl')
history_df = df_raw
target_users = history_df['customer_id'].unique().to_list()
make_chunked_submission_pkl(history_df, items_df, target_users, lgb_m, all_feats, output_file, chunk_size=EXPORT_CHUNK_SIZE, write_chunk_files=False)
